In [2]:

import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
import requests
from typing import List
load_dotenv()


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()

@dataclass
class Config:
    ALBERT_API_KEY: str = os.getenv("ALBERT_API_KEY", "")
    ALBERT_BASE_URL: str = "https://albert.api.etalab.gouv.fr/v1"
    LLM_MODEL: str = "albert-large"
    EMBEDDING_MODEL: str = "embeddings-small"
    RERANK_MODEL: str = "rerank-small"
    CHROMA_PERSIST_DIR: str = "./chroma_db"
    DOCUMENTS_DIR: str = "./documents"

In [3]:
ALBERT_API_KEY = os.getenv('ALBERT_API_KEY')
ALBERT_BASE_URL = "https://albert.api.etalab.gouv.fr/v1"

EMBEDDINGS_MODEL = "embeddings-small"
LLM_MODEL = "albert-large"
RERANK_MODEL = "rerank-small"

# Clients compatibles OpenAI
embeddings = OpenAIEmbeddings(
    model=EMBEDDINGS_MODEL,
    openai_api_key=ALBERT_API_KEY,
    openai_api_base=f"{ALBERT_BASE_URL}",
    model_kwargs={"encoding_format": "float"},
    chunk_size=50,
    max_retries=3,
    request_timeout=60

)
llm = ChatOpenAI(
    model=LLM_MODEL,
    openai_api_key=ALBERT_API_KEY,
    openai_api_base=f"{ALBERT_BASE_URL}",
    temperature=0.1
)


In [5]:
### CELLULE 4 FINALE: Indexation incrémentale intelligente
import os
import shutil
import json
import hashlib
from pathlib import Path
from datetime import datetime
import chromadb
from chromadb.config import Settings
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    UnstructuredMarkdownLoader,
    UnstructuredHTMLLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document


def load_odt_with_odfpy(file_path: str):
    """Charge ODT avec odfpy"""
    try:
        from odf import text, teletype
        from odf.opendocument import load
        
        textdoc = load(file_path)
        allparas = textdoc.getElementsByType(text.P)
        content = "\n".join([teletype.extractText(para) for para in allparas])
        
        return [Document(page_content=content, metadata={"source": file_path})]
    except ImportError:
        print(f"⚠️ odfpy non installé")
        return []
    except Exception as e:
        print(f"⚠️ Erreur ODT {Path(file_path).name}: {e}")
        return []


def load_document_safe(file_path: str):
    """Charge avec plusieurs tentatives et fallback"""
    ext = Path(file_path).suffix.lower()
    
    if ext == '.pdf':
        try:
            return PyPDFLoader(file_path).load()
        except Exception as e:
            print(f"⚠️ PDF erreur: {Path(file_path).name}")
            return []
    
    elif ext == '.docx':
        try:
            return Docx2txtLoader(file_path).load()
        except Exception as e:
            print(f"⚠️ DOCX erreur: {Path(file_path).name}")
            return []
    
    elif ext in ['.txt', '.text']:
        for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
            try:
                return TextLoader(file_path, encoding=encoding).load()
            except:
                continue
        print(f"⚠️ TXT encodage: {Path(file_path).name}")
        return []
    
    elif ext == '.md':
        try:
            return UnstructuredMarkdownLoader(file_path).load()
        except Exception as e:
            print(f"⚠️ MD erreur: {Path(file_path).name}")
            return []
    
    elif ext in ['.html', '.htm']:
        try:
            return UnstructuredHTMLLoader(file_path).load()
        except Exception as e:
            print(f"⚠️ HTML erreur: {Path(file_path).name}")
            return []
    
    elif ext == '.doc':
        print(f"⏭️ .doc ignoré (ancien format): {Path(file_path).name}")
        return []
    
    elif ext == '.odt':
        return load_odt_with_odfpy(file_path)
    
    return []


def get_file_hash(file_path: str) -> str:
    """Calcule le hash MD5 d'un fichier"""
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except:
        return ""


def get_file_metadata(file_path: str) -> dict:
    """Récupère métadonnées d'un fichier"""
    stat = os.stat(file_path)
    return {
        "path": file_path,
        "size": stat.st_size,
        "mtime": stat.st_mtime,
        "hash": get_file_hash(file_path)
    }


def load_index_metadata(db_path: str) -> dict:
    """Charge les métadonnées de l'index existant"""
    metadata_file = os.path.join(db_path, "index_metadata.json")
    if os.path.exists(metadata_file):
        try:
            with open(metadata_file, 'r') as f:
                return json.load(f)
        except:
            return {"files": {}, "last_update": None}
    return {"files": {}, "last_update": None}


def save_index_metadata(db_path: str, metadata: dict):
    """Sauvegarde les métadonnées de l'index"""
    metadata["last_update"] = datetime.now().isoformat()
    metadata_file = os.path.join(db_path, "index_metadata.json")
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)


def scan_documents_directory(directory: str):
    """Scanne le dossier et retourne les fichiers avec métadonnées"""
    SUPPORTED_EXTENSIONS = {'.pdf', '.docx', '.odt', '.txt', '.md', '.html', '.htm'}
    
    files_metadata = {}
    for root, dirs, files in os.walk(directory):
        for file in files:
            ext = Path(file).suffix.lower()
            if ext in SUPPORTED_EXTENSIONS:
                file_path = os.path.join(root, file)
                files_metadata[file_path] = get_file_metadata(file_path)
    
    return files_metadata


def incremental_indexing(documents_dir: str, db_path: str, embeddings, chunk_size=2000, chunk_overlap=400):
    """Indexation incrémentale : ajoute nouveaux, supprime disparus"""
    
    print(f"📁 Base: {db_path}")
    print(f"📂 Documents: {documents_dir}\n")
    
    # Créer dossier DB si n'existe pas
    os.makedirs(db_path, exist_ok=True)
    
    # Charger métadonnées existantes
    index_metadata = load_index_metadata(db_path)
    old_files = index_metadata.get("files", {})
    
    # Scanner dossier actuel
    current_files = scan_documents_directory(documents_dir)
    
    # Détecter changements
    new_files = []
    modified_files = []
    deleted_files = []
    unchanged_files = []
    
    for file_path, metadata in current_files.items():
        if file_path not in old_files:
            new_files.append(file_path)
        elif old_files[file_path].get("hash") != metadata.get("hash"):
            modified_files.append(file_path)
        else:
            unchanged_files.append(file_path)
    
    for file_path in old_files:
        if file_path not in current_files:
            deleted_files.append(file_path)
    
    # Afficher rapport
    print("📊 Analyse des changements:")
    print(f"   ✅ Nouveaux: {len(new_files)}")
    print(f"   🔄 Modifiés: {len(modified_files)}")
    print(f"   🗑️ Supprimés: {len(deleted_files)}")
    print(f"   ⏭️ Inchangés: {len(unchanged_files)}")
    
    # Si rien à faire
    if len(new_files) == 0 and len(modified_files) == 0 and len(deleted_files) == 0:
        print("\n✅ Aucun changement détecté, base à jour!")
        
        # Charger vectorstore existant
        client = chromadb.PersistentClient(path=db_path, settings=Settings(anonymized_telemetry=False))
        vectorstore = Chroma(
            client=client,
            collection_name="documents_rag",
            embedding_function=embeddings
        )
        retriever = vectorstore.as_retriever(search_kwargs={"k": 30})
        return vectorstore, retriever
    
    print(f"\n⏳ Mise à jour de l'index...\n")
    
    # Créer/charger client ChromaDB
    client = chromadb.PersistentClient(path=db_path, settings=Settings(anonymized_telemetry=False))
    
    collection_name = "documents_rag"
    
    # Récupérer ou créer collection
    try:
        collection = client.get_collection(name=collection_name)
        vectorstore = Chroma(
            client=client,
            collection_name=collection_name,
            embedding_function=embeddings
        )
        print("♻️ Collection existante chargée")
    except:
        vectorstore = Chroma(
            client=client,
            collection_name=collection_name,
            embedding_function=embeddings
        )
        print("🆕 Nouvelle collection créée")
    
    # Splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""]
    )
    
    # 1. SUPPRIMER documents disparus
    if deleted_files:
        print(f"\n🗑️ Suppression de {len(deleted_files)} fichiers disparus:")
        for file_path in deleted_files:
            print(f"   - {Path(file_path).name}")
            try:
                # Supprimer tous les chunks de ce fichier
                collection = client.get_collection(name=collection_name)
                results = collection.get(where={"source": file_path})
                if results and results['ids']:
                    collection.delete(ids=results['ids'])
                    print(f"     ✅ {len(results['ids'])} chunks supprimés")
            except Exception as e:
                print(f"     ⚠️ Erreur: {e}")
    
    # 2. SUPPRIMER puis RÉINDEXER documents modifiés
    files_to_index = new_files + modified_files
    
    if modified_files:
        print(f"\n🔄 Suppression versions anciennes de {len(modified_files)} fichiers modifiés:")
        for file_path in modified_files:
            print(f"   - {Path(file_path).name}")
            try:
                collection = client.get_collection(name=collection_name)
                results = collection.get(where={"source": file_path})
                if results and results['ids']:
                    collection.delete(ids=results['ids'])
                    print(f"     ✅ {len(results['ids'])} anciens chunks supprimés")
            except Exception as e:
                print(f"     ⚠️ Erreur: {e}")
    
    # 3. INDEXER nouveaux et modifiés
    if files_to_index:
        print(f"\n📝 Indexation de {len(files_to_index)} fichiers:")
        
        all_new_chunks = []
        for file_path in files_to_index:
            print(f"   ✅ {Path(file_path).name}")
            docs = load_document_safe(file_path)
            if docs:
                chunks = splitter.split_documents(docs)
                all_new_chunks.extend(chunks)
        
        print(f"\n⏳ Ajout de {len(all_new_chunks)} nouveaux chunks par batches de 50...")
        
        # Indexer par batches
        BATCH_SIZE = 50
        for i in range(0, len(all_new_chunks), BATCH_SIZE):
            batch = all_new_chunks[i:i+BATCH_SIZE]
            batch_num = (i // BATCH_SIZE) + 1
            total_batches = (len(all_new_chunks) + BATCH_SIZE - 1) // BATCH_SIZE
            
            print(f"Batch {batch_num}/{total_batches} ({len(batch)} chunks)...", end=" ")
            
            try:
                vectorstore.add_documents(batch)
                print("✅")
            except Exception as e:
                print(f"❌ {str(e)[:100]}")
    
    # 4. SAUVEGARDER métadonnées à jour
    index_metadata["files"] = current_files
    save_index_metadata(db_path, index_metadata)
    
    # Stats finales
    collection = client.get_collection(name=collection_name)
    total_chunks = collection.count()
    
    print(f"\n✅ Indexation terminée!")
    print(f"   Total fichiers: {len(current_files)}")
    print(f"   Total chunks: {total_chunks}")
    print(f"   Base: {db_path}")
    
    # Créer retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": 30})
    
    return vectorstore, retriever


# ✅ EXÉCUTION avec indexation incrémentale
DB_PATH = os.path.abspath("./chroma_db_rag")
DOCS_DIR = "documents/"

vectorstore, retriever = incremental_indexing(
    documents_dir=DOCS_DIR,
    db_path=DB_PATH,
    embeddings=embeddings,
    chunk_size=2000,
    chunk_overlap=400
)

print(f"\n🎉 Retriever prêt (top-30)")


📁 Base: /home/onyxia/work/chroma_db_rag
📂 Documents: documents/

📊 Analyse des changements:
   ✅ Nouveaux: 0
   🔄 Modifiés: 0
   🗑️ Supprimés: 1
   ⏭️ Inchangés: 6

⏳ Mise à jour de l'index...

♻️ Collection existante chargée

🗑️ Suppression de 1 fichiers disparus:
   - Note ZAN.odt
     ✅ 6 chunks supprimés

✅ Indexation terminée!
   Total fichiers: 6
   Total chunks: 195
   Base: /home/onyxia/work/chroma_db_rag

🎉 Retriever prêt (top-30)


In [6]:
# Cellule 5: Fonction HyDE
hyde_prompt = PromptTemplate.from_template("""
Question: {query}
Génère un document synthétique qui répondrait parfaitement à cette question.
Réponse structurée et factuelle:
""")

def generate_hyde_query(query: str):
    chain = hyde_prompt | llm
    hyde_doc = chain.invoke({"query": query}).content
    return f"{query} {hyde_doc}"  # Query enrichie pour meilleur embedding [web:3]

# Test
query = "Que peux tu dire du ZAN ?"
hyde_query = generate_hyde_query(query)
print("🧠 HyDE:", hyde_query[:300], "...")


🧠 HyDE: Que peux tu dire du ZAN ? **Document Synthétique : Le ZAN (Zéro Artificialisation Nette)**

### **1. Définition et Objectifs**
Le **Zéro Artificialisation Nette (ZAN)** est une politique publique française visant à **stopper l’artificialisation des sols** d’ici 2050. L’objectif est de **limiter la c ...


In [7]:
# Cellule 6: Rerank API Albert.ai
def rerank_documents(query: str, docs: List, top_k: int = 3):
    payload = {
        "model": RERANK_MODEL,
        "query": query,
        "documents": [doc.page_content for doc in docs],
        "top_n": top_k
    }
    headers = {"Authorization": f"Bearer {ALBERT_API_KEY}", "Content-Type": "application/json"}
    
    response = requests.post(f"{ALBERT_BASE_URL}/rerank", json=payload, headers=headers)
    if response.status_code == 200:
        results = response.json()["results"]
        return [docs[r["index"]] for r in results]
    return docs[:top_k]  # Fallback [web:1]

# Test rerank
docs_initial = retriever.invoke(hyde_query)
docs_reranked = rerank_documents(query, docs_initial)
print("📊 Reranked:", [doc.metadata.get('source', 'N/A') for doc in docs_reranked])


📊 Reranked: ['documents/Note ZAN.docx', 'documents/FichesExperimentationObjectifZAN_ADEMEnov2023.pdf', 'documents/FichesExperimentationObjectifZAN_ADEMEnov2023.pdf']


In [9]:
### CELLULE: Système de logging RAG
import sqlite3
import json
from datetime import datetime
from typing import List, Dict, Optional
from pathlib import Path

class RAGLogger:
    """Logger SQLite pour tracer toutes les interactions RAG"""
    
    def __init__(self, db_path: str = "./rag_logs.db"):
        self.db_path = db_path
        self._init_database()
    
    def _init_database(self):
        """Crée la table de logs si elle n'existe pas"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS rag_queries (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                user_query TEXT NOT NULL,
                hyde_query TEXT,
                retrieved_docs_count INTEGER,
                retrieved_docs_details TEXT,
                reranked_docs_count INTEGER,
                reranked_docs_details TEXT,
                final_answer TEXT,
                sources TEXT,
                rerank_scores TEXT,
                execution_time_seconds REAL,
                error TEXT
            )
        """)
        
        # Index pour recherches rapides
        cursor.execute("""
            CREATE INDEX IF NOT EXISTS idx_timestamp 
            ON rag_queries(timestamp)
        """)
        
        cursor.execute("""
            CREATE INDEX IF NOT EXISTS idx_user_query 
            ON rag_queries(user_query)
        """)
        
        conn.commit()
        conn.close()
        
        print(f"✅ Logger RAG initialisé: {self.db_path}")
    
    def log_query(
        self,
        user_query: str,
        hyde_query: Optional[str] = None,
        retrieved_docs: Optional[List] = None,
        reranked_docs: Optional[List] = None,
        final_answer: Optional[str] = None,
        sources: Optional[List[str]] = None,
        rerank_scores: Optional[List] = None,
        execution_time: Optional[float] = None,
        error: Optional[str] = None
    ) -> int:
        """
        Enregistre une requête RAG complète
        
        Returns:
            query_id: ID de la requête enregistrée
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Préparer les données JSON
        retrieved_docs_details = None
        if retrieved_docs:
            retrieved_docs_details = json.dumps([
                {
                    "content": doc.page_content[:200],  # Extrait
                    "source": doc.metadata.get("source", "N/A"),
                    "metadata": doc.metadata
                }
                for doc in retrieved_docs[:10]  # Limiter à 10 pour lisibilité
            ], ensure_ascii=False)
        
        reranked_docs_details = None
        if reranked_docs:
            reranked_docs_details = json.dumps([
                {
                    "content": doc.page_content[:200],
                    "source": doc.metadata.get("source", "N/A"),
                    "rerank_score": doc.metadata.get("rerank_score", None)
                }
                for doc in reranked_docs
            ], ensure_ascii=False)
        
        cursor.execute("""
            INSERT INTO rag_queries (
                timestamp,
                user_query,
                hyde_query,
                retrieved_docs_count,
                retrieved_docs_details,
                reranked_docs_count,
                reranked_docs_details,
                final_answer,
                sources,
                rerank_scores,
                execution_time_seconds,
                error
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            datetime.now().isoformat(),
            user_query,
            hyde_query,
            len(retrieved_docs) if retrieved_docs else 0,
            retrieved_docs_details,
            len(reranked_docs) if reranked_docs else 0,
            reranked_docs_details,
            final_answer,
            json.dumps(sources, ensure_ascii=False) if sources else None,
            json.dumps(rerank_scores, ensure_ascii=False) if rerank_scores else None,
            execution_time,
            error
        ))
        
        query_id = cursor.lastrowid
        conn.commit()
        conn.close()
        
        return query_id
    
    def get_recent_queries(self, limit: int = 10) -> List[Dict]:
        """Récupère les N dernières requêtes"""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT * FROM rag_queries 
            ORDER BY timestamp DESC 
            LIMIT ?
        """, (limit,))
        
        results = [dict(row) for row in cursor.fetchall()]
        conn.close()
        
        return results
    
    def search_queries(self, search_term: str, limit: int = 20) -> List[Dict]:
        """Recherche dans les questions ou réponses"""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT * FROM rag_queries 
            WHERE user_query LIKE ? OR final_answer LIKE ?
            ORDER BY timestamp DESC 
            LIMIT ?
        """, (f"%{search_term}%", f"%{search_term}%", limit))
        
        results = [dict(row) for row in cursor.fetchall()]
        conn.close()
        
        return results
    
    def get_stats(self) -> Dict:
        """Statistiques globales sur les requêtes"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Total requêtes
        cursor.execute("SELECT COUNT(*) FROM rag_queries")
        total_queries = cursor.fetchone()[0]
        
        # Requêtes avec erreurs
        cursor.execute("SELECT COUNT(*) FROM rag_queries WHERE error IS NOT NULL")
        error_queries = cursor.fetchone()[0]
        
        # Temps d'exécution moyen
        cursor.execute("SELECT AVG(execution_time_seconds) FROM rag_queries WHERE execution_time_seconds IS NOT NULL")
        avg_time = cursor.fetchone()[0]
        
        # Documents moyens récupérés
        cursor.execute("SELECT AVG(retrieved_docs_count) FROM rag_queries")
        avg_docs = cursor.fetchone()[0]
        
        # Sources les plus utilisées
        cursor.execute("""
            SELECT sources, COUNT(*) as count 
            FROM rag_queries 
            WHERE sources IS NOT NULL 
            GROUP BY sources 
            ORDER BY count DESC 
            LIMIT 5
        """)
        top_sources = cursor.fetchall()
        
        conn.close()
        
        return {
            "total_queries": total_queries,
            "error_queries": error_queries,
            "success_rate": (total_queries - error_queries) / total_queries * 100 if total_queries > 0 else 0,
            "avg_execution_time": avg_time,
            "avg_docs_retrieved": avg_docs,
            "top_sources": top_sources
        }
    
    def export_to_csv(self, output_path: str = "./rag_logs.csv"):
        """Exporte les logs en CSV pour analyse"""
        import csv
        
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("SELECT * FROM rag_queries ORDER BY timestamp DESC")
        rows = cursor.fetchall()
        
        # Récupérer noms de colonnes
        column_names = [description[0] for description in cursor.description]
        
        with open(output_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(column_names)
            writer.writerows(rows)
        
        conn.close()
        print(f"✅ Logs exportés vers {output_path}")


# Initialiser le logger
rag_logger = RAGLogger(db_path="./rag_logs.db")


✅ Logger RAG initialisé: ./rag_logs.db


In [10]:
### CELLULE: RAG avec logging automatique
import time
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def rag_hyde_rerank_with_logging(query: str, top_k_docs=5):
    """Pipeline RAG complet avec logging automatique"""
    
    start_time = time.time()
    error = None
    
    try:
        print(f"🔍 Query: {query}")
        
        # 1. Étape HyDE
        print("⏳ 1/4: Génération HyDE...")
        hyde_prompt = PromptTemplate.from_template(
            """Question: {query}
            
Génère un document détaillé qui répondrait à cette question:"""
        )
        hyde_chain = hyde_prompt | llm | StrOutputParser()
        hyde_doc = hyde_chain.invoke({"query": query})
        hyde_query = f"{query} {hyde_doc}"
        
        # 2. Retrieval initial
        print("⏳ 2/4: Retrieval initial...")
        docs_initial = retriever.invoke(hyde_query)
        
        # 3. Reranking (si disponible)
        print("⏳ 3/4: Rerank...")
        try:
            # Utiliser votre fonction rerank (ou compressed_retriever)
            docs_final = rerank_documents(query, docs_initial, top_k_docs)
        except Exception as e:
            print(f"⚠️ Rerank non disponible, utilisation top-{top_k_docs}")
            docs_final = docs_initial[:top_k_docs]
        
        # 4. Génération finale
        print("⏳ 4/4: Génération réponse...")
        context = "\n\n".join([d.page_content for d in docs_final])
        
        rag_prompt = PromptTemplate.from_template(
            """Contexte: {context}

Question: {query}

Réponds en utilisant UNIQUEMENT le contexte ci-dessus:"""
        )
        
        rag_chain = rag_prompt | llm | StrOutputParser()
        answer = rag_chain.invoke({"context": context, "query": query})
        
        # Extraire métadonnées
        sources = [d.metadata.get('source', 'N/A') for d in docs_final]
        rerank_scores = [d.metadata.get('rerank_score', None) for d in docs_final]
        
        execution_time = time.time() - start_time
        
        # ✅ LOGGER LA REQUÊTE
        query_id = rag_logger.log_query(
            user_query=query,
            hyde_query=hyde_query[:500] + "..." if len(hyde_query) > 500 else hyde_query,
            retrieved_docs=docs_initial,
            reranked_docs=docs_final,
            final_answer=answer,
            sources=sources,
            rerank_scores=rerank_scores,
            execution_time=execution_time,
            error=None
        )
        
        print(f"\n✅ Réponse générée en {execution_time:.2f}s (log #{query_id})")
        
        return {
            "query_id": query_id,
            "answer": answer,
            "sources": sources,
            "rerank_scores": rerank_scores,
            "hyde_query": hyde_query[:200] + "...",
            "n_docs_initial": len(docs_initial),
            "n_docs_final": len(docs_final),
            "execution_time": execution_time
        }
        
    except Exception as e:
        error = str(e)
        execution_time = time.time() - start_time
        
        # Logger l'erreur
        query_id = rag_logger.log_query(
            user_query=query,
            hyde_query=None,
            retrieved_docs=None,
            reranked_docs=None,
            final_answer=None,
            sources=None,
            rerank_scores=None,
            execution_time=execution_time,
            error=error
        )
        
        print(f"\n❌ Erreur (log #{query_id}): {error}")
        raise


# Fonction rerank simple (à adapter selon votre implémentation)
def rerank_documents(query: str, docs: List, top_k: int = 5):
    """Fonction rerank de base (remplacer par votre AlbertReranker)"""
    # Version simplifiée : retourne top-k
    return docs[:top_k]


# Test
result = rag_hyde_rerank_with_logging("parle moi de mad et moselle")
print(f"\n📄 Réponse: {result['answer'][:300]}...")
print(f"📁 Sources: {result['sources']}")


🔍 Query: Qparle moi de mad et moselle
⏳ 1/4: Génération HyDE...
⏳ 2/4: Retrieval initial...
⏳ 3/4: Rerank...
⏳ 4/4: Génération réponse...

✅ Réponse générée en 31.85s (log #1)

📄 Réponse: La Communauté de communes **Mad & Moselle** s'engage activement dans la démarche **Zéro Artificialisation Nette (ZAN)** à travers plusieurs actions concrètes :

1. **Inventaire des sites prioritaires** :
   - Elle a réalisé un **inventaire intercommunal** des surfaces à désimperméabiliser et désarti...
📁 Sources: ['documents/FichesExperimentationObjectifZAN_ADEMEnov2023.pdf', 'documents/note_Préfet situation PDCH.odt', 'documents/FichesExperimentationObjectifZAN_ADEMEnov2023.pdf', 'documents/note_Préfet situation PDCH.odt', 'documents/FichesExperimentationObjectifZAN_ADEMEnov2023.pdf']
